In [1]:
import psycopg2
import os
from dotenv import load_dotenv
load_dotenv()

conn =  psycopg2.connect(os.getenv("DATABASE_URL"))

In [2]:
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS songs (
    id SERIAL PRIMARY KEY,
    title TEXT NOT NULL,
    artist TEXT,
    duration FLOAT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS fingerprints (
    hash BIGINT NOT NULL,
    song_id INT NOT NULL,
    time_offset INT NOT NULL,
    anchor_freq INT NOT NULL,
    FOREIGN KEY (song_id) REFERENCES songs(id) ON DELETE CASCADE
);
""")

cur.execute("""
CREATE INDEX IF NOT EXISTS idx_fingerprint_hash
ON fingerprints(hash);
""")

cur.execute("""
CREATE INDEX IF NOT EXISTS idx_fingerprint_hash_anchor
ON fingerprints(hash, anchor_freq);
""")

conn.commit()
cur.close()
conn.close()

In [17]:
from utils import generate_hashes_for_audio
hashes = generate_hashes_for_audio("data/dafunk.mp3")

In [18]:
hashes

array([[5640198,       1,     697],
       [5742598,       1,     697],
       [6901766,       1,     697],
       ...,
       [7323653,    2255,     372],
       [7598085,    2255,     372],
       [8220677,    2255,     372]], shape=(19084, 3))

In [19]:
conn =  psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

cur.execute("""
    INSERT INTO songs (title, artist, duration)
    VALUES (%s, %s, %s)
    RETURNING id;
""", ("Da Funk", "Daft Punk", 52))

song_id = cur.fetchone()[0]

conn.commit()
cur.close()
conn.close()

In [20]:
from psycopg2.extras import execute_values

conn =  psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

data = [(int(h), int(song_id), int(offset), int(anchor_freq)) for h, offset, anchor_freq in hashes]

query = """
    INSERT INTO fingerprints (hash, song_id, time_offset, anchor_freq)
    VALUES %s
"""

execute_values(cur, query, data, page_size=10000)

conn.commit()
cur.close()
conn.close()

In [21]:
conn = psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

query_hashes = hashes[:, 0].astype(int).tolist()

if query_hashes:
    cur.execute(
        """
        SELECT hash, song_id, time_offset, anchor_freq
        FROM fingerprints
        WHERE hash = ANY(%s)
        """,
        (query_hashes,),
    )
    results = cur.fetchall()
else:
    results = []

cur.close()
conn.close()

results

[(4481037, 3, 111, 964),
 (4485132, 3, 1181, 963),
 (4497421, 3, 1366, 960),
 (4497421, 3, 1923, 960),
 (4526092, 1, 3756, 948),
 (4526092, 3, 716, 960),
 (4530188, 3, 1087, 959),
 (4558859, 3, 693, 946),
 (4567062, 3, 497, 943),
 (4603924, 3, 1984, 952),
 (4612108, 3, 1923, 960),
 (4616204, 3, 1366, 960),
 (4689941, 3, 1984, 952),
 (4743189, 3, 497, 943),
 (4747273, 1, 3603, 922),
 (4747273, 1, 8009, 895),
 (4747273, 3, 1181, 963),
 (4747274, 1, 7693, 897),
 (4747274, 3, 1366, 960),
 (4759563, 3, 1923, 960),
 (4780043, 3, 182, 892),
 (4788232, 3, 1518, 934),
 (4812809, 3, 1366, 960),
 (4821003, 3, 1181, 963),
 (4829211, 3, 497, 943),
 (4841493, 3, 2030, 910),
 (4849685, 3, 693, 946),
 (4878348, 3, 111, 964),
 (4882443, 1, 3875, 889),
 (4882443, 3, 473, 925),
 (4890645, 3, 1937, 908),
 (4898825, 1, 3539, 881),
 (4898825, 3, 1181, 963),
 (4902918, 1, 4980, 884),
 (4902918, 1, 7783, 859),
 (4902918, 1, 8338, 899),
 (4902918, 3, 1366, 960),
 (4911114, 1, 7890, 965),
 (4911114, 3, 182, 892

In [22]:
from collections import defaultdict, Counter

ANCHOR_TOL = 2

db_map = defaultdict(list)
for h, song_id, offset, anchor_freq in results:
    db_map[int(h)].append((int(song_id), int(offset), int(anchor_freq)))

votes = defaultdict(list)

for h, q_offset, q_anchor_freq in hashes:
    h = int(h)
    q_offset = int(q_offset)
    q_anchor_freq = int(q_anchor_freq)

    candidates = db_map.get(h, [])
    for song_id, db_offset, db_anchor_freq in candidates:
        if abs(db_anchor_freq - q_anchor_freq) > ANCHOR_TOL:
            continue
        delta = db_offset - q_offset
        votes[song_id].append(delta)

best_song = None
best_score = 0

for song_id, deltas in votes.items():
    counter = Counter(deltas)
    _, count = counter.most_common(1)[0]

    if count > best_score:
        best_score = count
        best_song = song_id

best_title = None
if best_song is not None:
    conn = psycopg2.connect(os.getenv("DATABASE_URL"))
    cur = conn.cursor()
    cur.execute("SELECT title FROM songs WHERE id = %s", (best_song,))
    row = cur.fetchone()
    best_title = row[0] if row else None
    cur.close()
    conn.close()

{"song_id": best_song, "title": best_title, "score": best_score, "anchor_tol": ANCHOR_TOL}

{'song_id': 3, 'title': 'Da Funk', 'score': 19084, 'anchor_tol': 2}

In [24]:
import os
import tempfile
from collections import defaultdict, Counter

import librosa
import numpy as np
import soundfile as sf
import psycopg2

from utils import generate_hashes_for_audio

def match_chunk_variant(audio_path, start_sec=30.0, chunk_sec=12.0, pitch_steps=0.0, gain=1.0, anchor_tol=2, min_score=10):
    y, sr = librosa.load(audio_path, sr=22050)

    start = int(start_sec * sr)
    end = min(len(y), start + int(chunk_sec * sr))
    if start >= len(y) or start >= end:
        return {
            "variant": {"start_sec": start_sec, "chunk_sec": chunk_sec, "pitch_steps": pitch_steps, "gain": gain},
            "error": "Invalid chunk range",
        }

    chunk = y[start:end]

    if pitch_steps != 0:
        chunk = librosa.effects.pitch_shift(chunk, sr=sr, n_steps=pitch_steps)

    if gain != 1.0:
        chunk = np.clip(chunk * gain, -1.0, 1.0)

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        temp_path = tmp.name
    sf.write(temp_path, chunk, sr)

    try:
        q_hashes = generate_hashes_for_audio(temp_path)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

    if q_hashes.size == 0:
        return {
            "variant": {"start_sec": start_sec, "chunk_sec": chunk_sec, "pitch_steps": pitch_steps, "gain": gain},
            "hash_count": 0,
            "match": None,
            "score": 0,
        }

    query_hashes = q_hashes[:, 0].astype(int).tolist()

    conn = psycopg2.connect(os.getenv("DATABASE_URL"))
    cur = conn.cursor()
    cur.execute(
        """
        SELECT hash, song_id, time_offset, anchor_freq
        FROM fingerprints
        WHERE hash = ANY(%s)
        """,
        (query_hashes,),
    )
    rows = cur.fetchall()
    cur.close()
    conn.close()

    db_map = defaultdict(list)
    for h, song_id, offset, anchor_freq in rows:
        db_map[int(h)].append((int(song_id), int(offset), int(anchor_freq)))

    votes = defaultdict(list)
    for h, q_offset, q_anchor_freq in q_hashes:
        h = int(h)
        q_offset = int(q_offset)
        q_anchor_freq = int(q_anchor_freq)

        candidates = db_map.get(h, [])
        for song_id, db_offset, db_anchor_freq in candidates:
            if abs(db_anchor_freq - q_anchor_freq) > anchor_tol:
                continue
            votes[song_id].append(db_offset - q_offset)

    best_song = None
    best_score = 0
    for song_id, deltas in votes.items():
        count = Counter(deltas).most_common(1)[0][1]
        if count > best_score:
            best_score = count
            best_song = song_id

    if best_score < min_score:
        best_song = None

    best_title = None
    if best_song is not None:
        conn = psycopg2.connect(os.getenv("DATABASE_URL"))
        cur = conn.cursor()
        cur.execute("SELECT title FROM songs WHERE id = %s", (best_song,))
        row = cur.fetchone()
        best_title = row[0] if row else None
        cur.close()
        conn.close()

    return {
        "variant": {
            "start_sec": start_sec,
            "chunk_sec": chunk_sec,
            "pitch_steps": pitch_steps,
            "gain": gain,
            "anchor_tol": anchor_tol,
            "min_score": min_score,
        },
        "hash_count": int(len(query_hashes)),
        "song_id": best_song,
        "title": best_title,
        "score": int(best_score),
    }

audio_path = "data/dafunk.mp3"
tests = [
    {"start_sec": 30, "chunk_sec": 12, "pitch_steps": 0, "gain": 1.0, "anchor_tol": 2, "min_score": 10},
    {"start_sec": 30, "chunk_sec": 12, "pitch_steps": 2, "gain": 1.0, "anchor_tol": 2, "min_score": 10},
    {"start_sec": 30, "chunk_sec": 12, "pitch_steps": -2, "gain": 1.0, "anchor_tol": 2, "min_score": 10},
    {"start_sec": 30, "chunk_sec": 12, "pitch_steps": 0, "gain": 0.6, "anchor_tol": 2, "min_score": 10},
    {"start_sec": 30, "chunk_sec": 12, "pitch_steps": 0, "gain": 1.4, "anchor_tol": 2, "min_score": 10},
]

outputs = [match_chunk_variant(audio_path, **cfg) for cfg in tests]
outputs

[{'variant': {'start_sec': 30,
   'chunk_sec': 12,
   'pitch_steps': 0,
   'gain': 1.0,
   'anchor_tol': 2,
   'min_score': 10},
  'hash_count': 4467,
  'song_id': 3,
  'title': 'Da Funk',
  'score': 4100},
 {'variant': {'start_sec': 30,
   'chunk_sec': 12,
   'pitch_steps': 2,
   'gain': 1.0,
   'anchor_tol': 2,
   'min_score': 10},
  'hash_count': 5501,
  'song_id': None,
  'title': None,
  'score': 3},
 {'variant': {'start_sec': 30,
   'chunk_sec': 12,
   'pitch_steps': -2,
   'gain': 1.0,
   'anchor_tol': 2,
   'min_score': 10},
  'hash_count': 5252,
  'song_id': None,
  'title': None,
  'score': 3},
 {'variant': {'start_sec': 30,
   'chunk_sec': 12,
   'pitch_steps': 0,
   'gain': 0.6,
   'anchor_tol': 2,
   'min_score': 10},
  'hash_count': 4461,
  'song_id': 3,
  'title': 'Da Funk',
  'score': 4133},
 {'variant': {'start_sec': 30,
   'chunk_sec': 12,
   'pitch_steps': 0,
   'gain': 1.4,
   'anchor_tol': 2,
   'min_score': 10},
  'hash_count': 4671,
  'song_id': 3,
  'title': 'Da